In [ ]:
import os

# Backend
!pip install fastapi uvicorn spacy pymupdf python-docx
!pip install sentence-transformers deap scikit-learn
!pip install openai python-multipart pandas numpy

# Download spaCy model
!python -m spacy download en_core_web_lg

# Frontend
!npx create-react-app frontend

# Change directory to frontend and install dependencies
# Use '&&' to chain commands, ensuring the next command runs only if the previous one succeeded
# Note: This will create a 'frontend' directory in the current working directory
# and then install dependencies within it.
!cd frontend && npm install tailwindcss axios recharts lucide-react

# Optional: To ensure subsequent Python cells in the root directory can still find backend files,
# you might want to explicitly change back to the root or use full paths.
# For now, we assume subsequent cells will handle paths relative to their execution context or that this setup step completes and new cells are run separately.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴Need to install the following packages:
create-react-app@5.1.0
Ok to proceed? (y) 

In [ ]:
# backend/parser/resume_parser.py
import fitz  # PyMuPDF
import spacy
import re
from dataclasses import dataclass
from typing import List

nlp = spacy.load("en_core_web_lg")

@dataclass
class ParsedResume:
    candidate_id: str
    name: str
    email: str
    phone: str
    skills: List[str]
    experience_years: float
    education_level: str   # bachelor, master, phd
    job_titles: List[str]
    certifications: List[str]
    raw_text: str

def extract_text_from_pdf(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    return " ".join(page.get_text() for page in doc)

def extract_skills(text: str, skill_list: List[str]) -> List[str]:
    text_lower = text.lower()
    return [skill for skill in skill_list if skill.lower() in text_lower]

def extract_experience_years(text: str) -> float:
    # Pattern: "5 years", "3+ years", "2-4 years"
    patterns = [
        r'(\d+)\+?\s*years?\s*of\s*experience',
        r'(\d+)\+?\s*years?\s*experience',
        r'experience\s*of\s*(\d+)\+?\s*years?'
    ]
    years = []
    for pattern in patterns:
        matches = re.findall(pattern, text.lower())
        years.extend([float(m) for m in matches])
    return max(years) if years else 0.0

def extract_education_level(text: str) -> str:
    text_lower = text.lower()
    if any(w in text_lower for w in ["ph.d", "phd", "doctorate"]):
        return "phd"
    elif any(w in text_lower for w in ["master", "m.s.", "mtech", "m.tech", "mba"]):
        return "master"
    elif any(w in text_lower for w in ["bachelor", "b.s.", "b.tech", "btech", "b.e."]):
        return "bachelor"
    return "other"

def extract_job_titles(text: str) -> List[str]:
    common_titles = [
        "software engineer", "data scientist", "ml engineer",
        "backend developer", "frontend developer", "full stack",
        "product manager", "data analyst", "devops engineer",
        "ai engineer", "team lead", "senior engineer", "manager"
    ]
    text_lower = text.lower()
    return [t for t in common_titles if t in text_lower]

def parse_resume(pdf_path: str, skill_list: List[str], candidate_id: str) -> ParsedResume:
    text = extract_text_from_pdf(pdf_path)
    doc = nlp(text)

    # Extract name & email via regex + NER
    email = re.findall(r'[\w.-]+@[\w.-]+\.\w+', text)
    phone = re.findall(r'\+?\d[\d\s\-().]{7,}\d', text)
    names = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]

    return ParsedResume(
        candidate_id=candidate_id,
        name=names[0] if names else "Unknown",
        email=email[0] if email else "",
        phone=phone[0] if phone else "",
        skills=extract_skills(text, skill_list),
        experience_years=extract_experience_years(text),
        education_level=extract_education_level(text),
        job_titles=extract_job_titles(text),
        certifications=extract_certifications(text),
        raw_text=text
    )

def extract_certifications(text: str) -> List[str]:
    cert_keywords = ["aws certified", "google certified", "pmp", "cissp",
                     "tensorflow certificate", "azure certified", "oracle certified"]
    text_lower = text.lower()
    return [c for c in cert_keywords if c in text_lower]

In [ ]:
# backend/parser/jd_parser.py
from dataclasses import dataclass
from typing import List
import re

@dataclass
class ParsedJD:
    title: str
    required_skills: List[str]
    min_experience_years: float
    preferred_education: str
    required_titles: List[str]

def parse_job_description(jd_text: str, skill_list: List[str]) -> ParsedJD:
    text_lower = jd_text.lower()

    # Extract required skills
    skills = [s for s in skill_list if s.lower() in text_lower]

    # Extract minimum experience
    exp_patterns = [r'(\d+)\+?\s*years?\s*of\s*experience', r'minimum\s*(\d+)\s*years?']
    exp_years = []
    for pattern in exp_patterns:
        matches = re.findall(pattern, text_lower)
        exp_years.extend([float(m) for m in matches])

    # Extract education preference
    if "phd" in text_lower or "doctorate" in text_lower:
        education = "phd"
    elif "master" in text_lower or "mtech" in text_lower:
        education = "master"
    else:
        education = "bachelor"

    return ParsedJD(
        title=extract_title(jd_text),
        required_skills=skills,
        min_experience_years=min(exp_years) if exp_years else 0.0,
        preferred_education=education,
        required_titles=[]
    )

def extract_title(jd_text: str) -> str:
    first_line = jd_text.strip().split('\n')[0]
    return first_line[:100]

In [ ]:
# backend/scoring/rule_engine.py
from dataclasses import dataclass
from typing import List, Dict
from sentence_transformers import SentenceTransformer, util
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

@dataclass
class ScoreBreakdown:
    candidate_id: str
    skills_score: float       # 0-100
    experience_score: float   # 0-100
    education_score: float    # 0-100
    title_score: float        # 0-100
    certification_score: float # 0-100
    semantic_score: float     # 0-100
    composite_score: float    # final weighted score
    priority_tier: str        # Tier 1 / Tier 2 / Tier 3
    knowledge_base_flags: List[str]  # special flags from KB

# ── WEIGHTS (will be optimized by GA in Step 3) ──
DEFAULT_WEIGHTS = {
    "skills": 0.35,
    "experience": 0.25,
    "education": 0.15,
    "title": 0.15,
    "certification": 0.10
}

def score_skills(resume_skills: List[str],
                 jd_skills: List[str]) -> float:
    if not jd_skills:
        return 50.0
    matched = len(set(resume_skills) & set(jd_skills))
    return (matched / len(jd_skills)) * 100

def score_experience(candidate_years: float,
                     required_years: float) -> float:
    if required_years == 0:
        return 80.0
    ratio = candidate_years / required_years
    if ratio >= 1.5:   return 100.0
    elif ratio >= 1.0: return 90.0
    elif ratio >= 0.8: return 70.0
    elif ratio >= 0.5: return 50.0
    else:              return 20.0

def score_education(candidate_edu: str,
                    required_edu: str) -> float:
    levels = {"other": 0, "bachelor": 1, "master": 2, "phd": 3}
    c = levels.get(candidate_edu, 0)
    r = levels.get(required_edu, 1)
    if c >= r:     return 100.0
    elif c == r-1: return 70.0
    else:          return 40.0

def score_titles(candidate_titles: List[str],
                 jd_title: str) -> float:
    if not candidate_titles:
        return 0.0
    jd_emb = model.encode(jd_title, convert_to_tensor=True)
    title_embs = model.encode(candidate_titles, convert_to_tensor=True)
    similarities = util.cos_sim(jd_emb, title_embs)[0]
    return float(similarities.max()) * 100

def score_semantic(resume_text: str, jd_text: str) -> float:
    # Semantic similarity between full resume and JD
    r_emb = model.encode(resume_text[:512], convert_to_tensor=True)
    j_emb = model.encode(jd_text[:512], convert_to_tensor=True)
    return float(util.cos_sim(r_emb, j_emb)[0][0]) * 100

def score_certifications(candidate_certs: List[str],
                         jd_text: str) -> float:
    if not candidate_certs:
        return 0.0
    return min(len(candidate_certs) * 20, 100.0)

# ── KNOWLEDGE BASE RULES ──
def apply_knowledge_base(resume, jd, breakdown) -> List[str]:
    flags = []

    # KB Rule 1: High potential career switcher
    skill_overlap = len(set(resume.skills) & set(jd.required_skills))
    if resume.experience_years >= 3 and skill_overlap >= 3:
        if not any(t in resume.job_titles for t in jd.required_titles):
            flags.append("⭐ High Potential — Strong transferable skills")

    # KB Rule 2: Overqualified but valuable
    if resume.experience_years >= jd.min_experience_years * 2:
        flags.append("🎯 Senior Profile — Exceeds experience requirement")

    # KB Rule 3: Leadership signal
    leadership_titles = ["lead", "manager", "head", "director", "senior"]
    if any(l in " ".join(resume.job_titles) for l in leadership_titles):
        if "lead" in jd.title.lower() or "senior" in jd.title.lower():
            flags.append("👑 Leadership Match — Past leadership aligns with role")

    # KB Rule 4: Perfect education + skills combo
    edu_levels = {"bachelor": 1, "master": 2, "phd": 3}
    if (edu_levels.get(resume.education_level, 0) >=
            edu_levels.get(jd.preferred_education, 1)
            and breakdown.skills_score >= 80):
        flags.append("✅ Strong Academic + Skills Fit")

    return flags

def compute_composite(breakdown: ScoreBreakdown,
                      weights: Dict = None) -> float:
    w = weights or DEFAULT_WEIGHTS
    return (
        breakdown.skills_score       * w["skills"] +
        breakdown.experience_score   * w["experience"] +
        breakdown.education_score    * w["education"] +
        breakdown.title_score        * w["title"] +
        breakdown.certification_score * w["certification"]
    )

def assign_priority_tier(score: float) -> str:
    if score >= 75:   return "Tier 1 — Strong Fit"
    elif score >= 55: return "Tier 2 — Good Fit"
    else:             return "Tier 3 — Potential Fit"

def score_resume(resume, jd, weights=None) -> ScoreBreakdown:
    bd = ScoreBreakdown(
        candidate_id=resume.candidate_id,
        skills_score=score_skills(resume.skills, jd.required_skills),
        experience_score=score_experience(resume.experience_years,
                                           jd.min_experience_years),
        education_score=score_education(resume.education_level,
                                         jd.preferred_education),
        title_score=score_titles(resume.job_titles, jd.title),
        certification_score=score_certifications(resume.certifications,
                                                  jd.title),
        semantic_score=score_semantic(resume.raw_text, jd.title),
        composite_score=0.0,
        priority_tier="",
        knowledge_base_flags=[]
    )
    bd.composite_score = compute_composite(bd, weights)
    bd.priority_tier = assign_priority_tier(bd.composite_score)
    bd.knowledge_base_flags = apply_knowledge_base(resume, jd, bd)
    return bd

In [ ]:
# backend/optimization/genetic_algorithm.py
import random
import numpy as np
from deap import base, creator, tools, algorithms
from typing import List, Dict

# ── GENETIC ALGORITHM TO OPTIMIZE SCORING WEIGHTS ──

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

def create_weights():
    # 5 weights that must sum to 1.0
    w = [random.random() for _ in range(5)]
    total = sum(w)
    return [x / total for x in w]

toolbox.register("individual", tools.initIterate,
                 creator.Individual, create_weights)
toolbox.register("population", tools.initRepeat,
                 list, toolbox.individual)

def fitness_function(individual, resumes, jd,
                     ground_truth_ranking):
    """
    Evaluate how well this weight combo ranks
    candidates vs a known good ranking.
    ground_truth_ranking: list of candidate_ids
    in ideal order (from recruiter feedback or
    historical data)
    """
    from scoring.rule_engine import score_resume, DEFAULT_WEIGHTS

    weights = {
        "skills":        individual[0],
        "experience":    individual[1],
        "education":     individual[2],
        "title":         individual[3],
        "certification": individual[4]
    }

    # Score all resumes with this weight combo
    scores = []
    for resume in resumes:
        bd = score_resume(resume, jd, weights)
        scores.append((resume.candidate_id, bd.composite_score))

    # Sort by score descending
    ranked = [cid for cid, _ in sorted(scores,
               key=lambda x: x[1], reverse=True)]

    # Kendall Tau correlation with ground truth
    score = kendall_tau_similarity(ranked, ground_truth_ranking)
    return (score,)

def kendall_tau_similarity(ranking1, ranking2):
    """Measures how similar two rankings are (0 to 1)"""
    n = len(ranking1)
    if n <= 1:
        return 1.0
    concordant = 0
    discordant = 0
    pos2 = {v: i for i, v in enumerate(ranking2)}
    for i in range(n):
        for j in range(i + 1, n):
            if ranking1[i] in pos2 and ranking1[j] in pos2:
                d = (pos2[ranking1[i]] - pos2[ranking1[j]])
                if d > 0:
                    concordant += 1
                else:
                    discordant += 1
    total = concordant + discordant
    return concordant / total if total > 0 else 0.5

def normalize_weights(individual):
    total = sum(individual)
    return [x / total for x in individual]

toolbox.register("mate", tools.cxBlend, alpha=0.3)
toolbox.register("mutate", tools.mutGaussian,
                 mu=0, sigma=0.1, indpb=0.3)
toolbox.register("select", tools.selTournament, tournsize=3)

def run_genetic_algorithm(resumes, jd,
                          ground_truth, generations=50):
    toolbox.register("evaluate", fitness_function,
                     resumes=resumes, jd=jd,
                     ground_truth_ranking=ground_truth)

    population = toolbox.population(n=50)
    hof = tools.HallOfFame(1)

    algorithms.eaSimple(population, toolbox,
                        cxpb=0.7, mutpb=0.2,
                        ngen=generations, halloffame=hof,
                        verbose=False)

    best = normalize_weights(hof[0])
    return {
        "skills":        best[0],
        "experience":    best[1],
        "education":     best[2],
        "title":         best[3],
        "certification": best[4]
    }

In [ ]:
# backend/optimization/astar_ranking.py
import heapq
from typing import List, Tuple

# ── A* INFORMED SEARCH FOR OPTIMAL CANDIDATE RANKING ──

class CandidateNode:
    def __init__(self, candidate_id, score, index):
        self.candidate_id = candidate_id
        self.score = score
        self.index = index

    def __lt__(self, other):
        # Max-heap via negation
        return self.score > other.score

def astar_rank_candidates(score_breakdowns) -> List[str]:
    """
    Uses A* (priority queue) to produce optimal
    ranked list of candidates by composite score.
    Heuristic h(n) = composite score from rule engine.
    """
    heap = []
    for bd in score_breakdowns:
        node = CandidateNode(
            bd.candidate_id,
            bd.composite_score,
            len(heap)
        )
        heapq.heappush(heap, node)

    ranked = []
    while heap:
        node = heapq.heappop(heap)
        ranked.append(node.candidate_id)

    return ranked

def get_ranked_results(score_breakdowns,
                       resumes_map: dict) -> List[dict]:
    ranked_ids = astar_rank_candidates(score_breakdowns)
    scores_map = {bd.candidate_id: bd
                  for bd in score_breakdowns}

    results = []
    for rank, cid in enumerate(ranked_ids, 1):
        bd = scores_map[cid]
        resume = resumes_map[cid]
        results.append({
            "rank": rank,
            "candidate_id": cid,
            "name": resume.name,
            "email": resume.email,
            "composite_score": round(bd.composite_score, 1),
            "priority_tier": bd.priority_tier,
            "skills_score": round(bd.skills_score, 1),
            "experience_score": round(bd.experience_score, 1),
            "education_score": round(bd.education_score, 1),
            "title_score": round(bd.title_score, 1),
            "knowledge_base_flags": bd.knowledge_base_flags,
            "matched_skills": list(
                set(resume.skills) &
                set([])  # fill with jd.required_skills
            )
        })
    return results

In [ ]:
# backend/justification/llm_justifier.py
import anthropic

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var

def generate_justification(candidate_result: dict,
                            jd_title: str,
                            jd_skills: list) -> str:
    prompt = f"""
You are an expert HR analyst. Write a professional,
positive justification (3-4 sentences) for why this
candidate is suitable for the role.

Role: {jd_title}
Required Skills: {', '.join(jd_skills)}

Candidate: {candidate_result['name']}
Overall Score: {candidate_result['composite_score']}/100
Priority Tier: {candidate_result['priority_tier']}
Skills Score: {candidate_result['skills_score']}/100
Experience Score: {candidate_result['experience_score']}/100
Education Score: {candidate_result['education_score']}/100
Special Flags: {', '.join(candidate_result['knowledge_base_flags'])}

Write a positive, factual justification explaining
why this candidate is a good fit. Do NOT say they
are rejected or unsuitable. Focus on their strengths
and potential contribution.
"""
    message = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

In [ ]:
# backend/main.py
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from typing import List
import tempfile, os, uuid

app = FastAPI(title="Resume Intelligence Platform")

app.add_middleware(CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_methods=["*"], allow_headers=["*"])

SKILL_LIST = [
    "python", "java", "javascript", "react", "nodejs",
    "machine learning", "deep learning", "sql", "mongodb",
    "aws", "docker", "kubernetes", "tensorflow", "pytorch",
    "data analysis", "nlp", "computer vision", "agile", "git"
]

@app.post("/api/analyze")
async def analyze_resumes(
    jd_text: str = Form(...),
    resumes: List[UploadFile] = File(...)
):
    from parser.resume_parser import parse_resume
    from parser.jd_parser import parse_job_description
    from scoring.rule_engine import score_resume
    from optimization.astar_ranking import get_ranked_results
    from justification.llm_justifier import generate_justification

    # 1. Parse JD
    jd = parse_job_description(jd_text, SKILL_LIST)

    # 2. Parse all resumes
    parsed_resumes = []
    resumes_map = {}
    for resume_file in resumes:
        cid = str(uuid.uuid4())[:8]
        with tempfile.NamedTemporaryFile(
                suffix=".pdf", delete=False) as tmp:
            tmp.write(await resume_file.read())
            tmp_path = tmp.name
        parsed = parse_resume(tmp_path, SKILL_LIST, cid)
        parsed_resumes.append(parsed)
        resumes_map[cid] = parsed
        os.unlink(tmp_path)

    # 3. Score all resumes
    score_breakdowns = [
        score_resume(r, jd) for r in parsed_resumes
    ]

    # 4. Rank using A*
    ranked_results = get_ranked_results(
        score_breakdowns, resumes_map
    )

    # 5. Generate LLM justifications
    for result in ranked_results:
        result["justification"] = generate_justification(
            result, jd.title, jd.required_skills
        )

    return {
        "job_title": jd.title,
        "required_skills": jd.required_skills,
        "total_candidates": len(ranked_results),
        "candidates": ranked_results
    }

@app.get("/api/health")
def health(): return {"status": "ok"}

In [ ]:
// frontend/src/App.jsx
import { useState } from "react";
import UploadPage from "./pages/UploadPage";
import DashboardPage from "./pages/DashboardPage";

export default function App() {
  const [results, setResults] = useState(null);
  return results
    ? <DashboardPage results={results} onReset={() => setResults(null)} />
    : <UploadPage onResults={setResults} />;
}

In [ ]:
// frontend/src/pages/UploadPage.jsx
import { useState } from "react";
import axios from "axios";

export default function UploadPage({ onResults }) {
  const [jd, setJd] = useState("");
  const [files, setFiles] = useState([]);
  const [loading, setLoading] = useState(false);

  const handleSubmit = async () => {
    setLoading(true);
    const form = new FormData();
    form.append("jd_text", jd);
    files.forEach(f => form.append("resumes", f));
    const res = await axios.post(
      "http://localhost:8000/api/analyze", form
    );
    onResults(res.data);
    setLoading(false);
  };

  return (
    <div className="min-h-screen bg-gray-50 flex items-center justify-center p-8">
      <div className="bg-white rounded-2xl shadow-lg p-8 w-full max-w-2xl">
        <h1 className="text-3xl font-bold text-blue-700 mb-2">
          🎯 Resume Intelligence Platform
        </h1>
        <p className="text-gray-500 mb-6">
          Upload resumes and a job description to get AI-powered scoring
        </p>
        <textarea
          className="w-full border rounded-xl p-4 h-40 mb-4 text-sm"
          placeholder="Paste your Job Description here..."
          value={jd} onChange={e => setJd(e.target.value)}
        />
        <input type="file" multiple accept=".pdf,.docx"
          className="mb-4 w-full"
          onChange={e => setFiles(Array.from(e.target.files))}
        />
        <p className="text-sm text-gray-400 mb-4">
          {files.length} resume(s) selected
        </p>
        <button
          onClick={handleSubmit}
          disabled={!jd || !files.length || loading}
          className="w-full bg-blue-600 text-white py-3 rounded-xl font-semibold hover:bg-blue-700 disabled:opacity-50"
        >
          {loading ? "Analyzing..." : "Analyze Resumes →"}
        </button>
      </div>
    </div>
  );
}

In [ ]:
// frontend/src/pages/DashboardPage.jsx
import CandidateCard from "../components/CandidateCard";
import ScoreChart from "../components/ScoreChart";

export default function DashboardPage({ results, onReset }) {
  const tierColors = {
    "Tier 1 — Strong Fit": "bg-green-100 text-green-700 border-green-300",
    "Tier 2 — Good Fit":   "bg-blue-100 text-blue-700 border-blue-300",
    "Tier 3 — Potential Fit": "bg-yellow-100 text-yellow-700 border-yellow-300"
  };

  return (
    <div className="min-h-screen bg-gray-50 p-8">
      <div className="max-w-6xl mx-auto">
        <div className="flex justify-between items-center mb-6">
          <div>
            <h1 className="text-3xl font-bold text-gray-800">
              📊 Candidate Dashboard
            </h1>
            <p className="text-gray-500">
              {results.job_title} — {results.total_candidates} candidates ranked
            </p>
          </div>
          <button onClick={onReset}
            className="bg-gray-200 px-4 py-2 rounded-lg hover:bg-gray-300">
            ← New Analysis
          </button>
        </div>

        <ScoreChart candidates={results.candidates} />

        <div className="grid gap-4 mt-6">
          {results.candidates.map(c => (
            <CandidateCard key={c.candidate_id}
              candidate={c} tierColors={tierColors} />
          ))}
        </div>
      </div>
    </div>
  );
}

In [ ]:
// frontend/src/components/CandidateCard.jsx
import { useState } from "react";

export default function CandidateCard({ candidate, tierColors }) {
  const [expanded, setExpanded] = useState(false);
  const c = candidate;

  return (
    <div className="bg-white rounded-xl shadow p-6 border border-gray-100">
      <div className="flex justify-between items-start">
        <div>
          <div className="flex items-center gap-3">
            <span className="text-2xl font-bold text-gray-300">
              #{c.rank}
            </span>
            <div>
              <h3 className="text-lg font-bold text-gray-800">{c.name}</h3>
              <p className="text-gray-500 text-sm">{c.email}</p>
            </div>
          </div>
          <div className="flex gap-2 mt-2 flex-wrap">
            <span className={`text-xs px-3 py-1 rounded-full border font-medium ${tierColors[c.priority_tier]}`}>
              {c.priority_tier}
            </span>
            {c.knowledge_base_flags.map((f, i) => (
              <span key={i} className="text-xs px-2 py-1 bg-purple-50 text-purple-700 rounded-full border border-purple-200">
                {f}
              </span>
            ))}
          </div>
        </div>

        {/* Score Circle */}
        <div className="text-center">
          <div className="w-16 h-16 rounded-full bg-blue-600 flex items-center justify-center">
            <span className="text-white font-bold text-lg">
              {c.composite_score}
            </span>
          </div>
          <p className="text-xs text-gray-400 mt-1">Score</p>
        </div>
      </div>

      {/* Score Bars */}
      <div className="mt-4 grid grid-cols-2 gap-3">
        {[
          ["Skills", c.skills_score, "bg-blue-500"],
          ["Experience", c.experience_score, "bg-green-500"],
          ["Education", c.education_score, "bg-purple-500"],
          ["Title Match", c.title_score, "bg-orange-500"],
        ].map(([label, score, color]) => (
          <div key={label}>
            <div className="flex justify-between text-xs text-gray-500 mb-1">
              <span>{label}</span><span>{Math.round(score)}%</span>
            </div>
            <div className="h-2 bg-gray-100 rounded-full">
              <div className={`h-2 ${color} rounded-full`}
                style={{ width: `${score}%` }} />
            </div>
          </div>
        ))}
      </div>

      {/* Justification */}
      <button onClick={() => setExpanded(!expanded)}
        className="mt-4 text-blue-600 text-sm hover:underline">
        {expanded ? "Hide" : "Show"} AI Justification ▾
      </button>
      {expanded && (
        <div className="mt-3 p-4 bg-blue-50 rounded-lg text-sm text-gray-700 border border-blue-100">
          💡 {c.justification}
        </div>
      )}
    </div>
  );
}